In [1]:
import torch
from torch.nn import Module, ModuleList, Parameter, Buffer
import tiktoken
import math
import os
import re
import numpy as np
from collections import Counter
print('Hello World')

Hello World


#### ***15. Generation***
Finally, write a function that will sequentially generate a response from the LLM sequentially given a prompt.  The process should work as follows:
1. First, run the model on the full prompt to predict the next output token. Then sample from the softmax distribution applied to this distribution, scaled by dividing by `temp`.
2. Repeatedly append this token the sequence, re-run the model (using KV cache) to predict the distribution over next tokens and sample from the temperature-scaled softmax distribution.
3. If `verbose` is True, after each generated token, print it's string representation, using the call `tokenizer.decode(token)`.  Additionally, check if each generated token is in the set `tokenizer.stop_tokens`, and break out of the generation loop if so (even if the system has not generated `max_tokens` tokens).

Some extra hints / stuff you would come across:
- the tokenizer accepts a list of tokens,
- this is optional, but to print print the output properly with `verbose`, you should add an `end=True, flush=true` argument to your print statements.


- Last of all:
4. You have be sure to initailize the input tokens with the `device` keyword set to the same device as the model (e.g., obtained by `next(model.parameters()).device`)
5. The tiktoken tokenizer has a field `tokenizer.eot_token` that is output when the text ends.  After this happens you should break.

### Temperature Sampling ###

In [3]:
logits = torch.tensor([[2.0, 1.0, 0.1, -1.0]])
temp = 0.7
prob = torch.softmax(logits / temp, dim=-1) 
### prob ≈ [0.62, 0.23, 0.10, 0.05]
print(torch.multinomial(prob, 1)) # 1 means: For each batch row, pick 1 token according to probability distribution (prob)

tensor([[1]])


In [4]:
def generate(model, prompt_tokens, tokenizer, temp=0.7, max_tokens=500, verbose=True):
    """
    Autoregressively sample tokens from a language model using its KV cache.
    Inputs:
        model: Module - language model mapping token sequences to logits
        prompt_tokens: list[int] - initial prompt tokens
        tokenizer: object - tokenizer with decode() and stop_tokens
        temp: float - sampling temperature
        max_tokens: int - max number of new tokens to generate
        verbose: bool - whether to print each generated token as it is sampled
    Output:
        list[int] - generated tokens, excluding the prompt tokens
    """
    device = next(model.parameters()).device
    ### Our model expect tokens in torch.Tensor[int] (batch_size x seq_len) and .reshape(1,-1) add that batch_size - input token ids
    tokens = torch.tensor(prompt_tokens, device=device).reshape(1,-1) ### ensures your model and input token tensor set to the same device (CPU or GPU) 
    logits = model(tokens, seq_pos=0, use_kv_cache=True) ### logits.shape => (batch_size, seq_len, num_tokens) #tokens start from position 0
    seq_pos = tokens.shape[1] #now update the seq_pos i.e. go to the last position of seq_len and start from there

    output = [] ### This will not include prompt tokens, only the generated tokens.

    ### for each batch size the model will go through each sequence (row) and generete new tokens (columns) which will append then to each row.
    ### before, (B,T,V) => (B,T,V+new_generated_tokens)
    for _ in range(max_tokens):
        ### Temperature sampling: P = softmax(logits_T), logits_T - the last row of the logits
        prob = torch.softmax(logits[:,-1,:]/temp, dim=-1) ### for all the batches take the last seq
        ### now let's generate tensor samples which belongs to this probability distribution 
        next_tokens = torch.multinomial(prob, 1) 
        token_id = next_tokens.item() # Returns the value of this tensor as a standard Python number
        output.append(token_id)

        if verbose:
            print(tokenizer.decode([token_id]), end="", flush=True)
        if token_id == tokenizer.eot_token: # you will see this: "<|endoftext|>", once generating text is done.
            break

        logits = model(next_tokens, seq_pos, use_kv_cache=True) # we only pass the new token not the entire token
        ### Y_T+1 = LLM(X_T+1)
        seq_pos += 1 # Now again the seq_pos will start internally from tokens.shape[1]+1, ...then again tokens.shape[1]+1+1
    
    return output